# 02b · Attention-MIL training  [GPU]
Replaces slice cleaning + voting with **attention-based Multiple-Instance Learning**: each fruit (bag) = its 72 slices; a frozen backbone extracts per-slice features (cached once), and a gated-attention head learns which slices matter and outputs one probability per fruit. Trains all four backbones.

Run after `01_partitioning`. No slice curation needed — skip `02a`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec, mil
CFG.out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
dl_seed=CFG.seed
import random, os; os.environ['PYTHONHASHSEED']=str(dl_seed); random.seed(dl_seed); np.random.seed(dl_seed)
import tensorflow as tf; tf.random.set_seed(dl_seed)
fruits = ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
by={f.fruit_id:f for f in fruits}
sp=pd.read_csv(CFG.out_dir/'split_single.csv').set_index('fruit_id')['split']
tr=list(sp[sp=='train'].index); va=list(sp[sp=='val'].index); te=list(sp[sp=='test'].index)
ylab={i:by[i].label for i in by}
print(f'fruits: train={len(tr)} val={len(va)} test={len(te)}')

## Step 1 — extract & cache per-slice features (once per backbone)

In [ ]:
predictions={}
for bb in CFG.final_backbones:
    print(f'\n===== {bb} =====')
    feats = mil.extract_features(CFG, bb, fruits)           # cached to features_<bb>.npz
    Xtr=mil.stack_bags(feats,tr); Xva=mil.stack_bags(feats,va); Xte=mil.stack_bags(feats,te)
    ytr=np.array([ylab[i] for i in tr]); yva=np.array([ylab[i] for i in va]); yte=np.array([ylab[i] for i in te])
    print(f'  bag tensors: Xtr{Xtr.shape} Xva{Xva.shape} Xte{Xte.shape}')
    model = mil.train_mil(CFG, Xtr,ytr, Xva,yva, verbose=2)
    try:
        model.save(CFG.out_dir/f'mil_{bb}.keras')
    except Exception as e:
        model.save_weights(str(CFG.out_dir/f'mil_{bb}.weights.h5'))
        print('  (saved weights only; full save skipped:', type(e).__name__, ')')
    pva,ava = mil.predict_bags(model, Xva)
    pte,ate = mil.predict_bags(model, Xte)
    predictions[bb]={'val':{'ids':va,'y':yva,'probs':pva},
                     'test':{'ids':te,'y':yte,'probs':pte,'attn':ate}}
    tf.keras.backend.clear_session()


## Step 2 — save predictions for 03b

In [ ]:
import pickle
pickle.dump(predictions, open(CFG.out_dir/'mil_predictions.pkl','wb'))
print('saved mil_predictions.pkl for', list(predictions))

Note: the backbone is frozen (feature extractor). If the four models underfit (low val AUC), the next step is fine-tuning the last blocks end-to-end — tell me and I'll add that variant.